In [1]:
import sunpy
import sunpy.map
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d,RegularGridInterpolator
from astropy.coordinates import SkyCoord
from scipy.io import readsav
from matplotlib.patches import Rectangle
from scipy.ndimage import zoom
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord
import cv2
import glob
import os

In [2]:
#检索目录
hmi_path=r'D:\C'
file_name=os.listdir(hmi_path)
file_path=[]
for i in file_name:
    file_path.append(os.path.join(hmi_path,i,'hmi.B_720s'))

In [3]:
print(file_path[0])

D:\C\20120701_154100_UTC\hmi.B_720s


In [4]:
print(glob.glob(os.path.join(file_path[0], '*.field.fits')))

['D:\\C\\20120701_154100_UTC\\hmi.B_720s\\hmi.b_720s.20120701_153600_TAI.field.fits']


In [5]:
#计算生成Br，Bh, inclination_calc.fits
#M级耀斑前面有一部分是float64的，占用空间太大，后面改成32的了
cnt=0
for i in file_path[:]:
    cnt+=1
    try:
        print(cnt)
        d=readsav(os.path.join(i, 'bxyz.sav'))
        mapbx = d["mapbx"][0][0]
        mapby = d["mapby"][0][0]
        mapbz = d["mapbz"][0][0]
        hmi_path=i
        B_map_4098 = sunpy.map.Map(glob.glob(os.path.join(hmi_path, '*.field.fits'))[0]).rotate()
        B_map_data=B_map_4098.data
        B_map_data=cv2.resize(B_map_data,(4096,4096),interpolation=cv2.INTER_AREA)
        B_map=sunpy.map.Map(B_map_data,B_map_4098.meta)
        inc_map_4098 = sunpy.map.Map(glob.glob(os.path.join(hmi_path, '*.inclination.fits'))[0]).rotate()
        inc_map_data=inc_map_4098.data
        inc_map_data=cv2.resize(inc_map_data,(4096,4096),interpolation=cv2.INTER_AREA)
        inc_map=sunpy.map.Map(inc_map_data,inc_map_4098.meta)
        azi_map_4098 = sunpy.map.Map(glob.glob(os.path.join(hmi_path, '*.azimuth.fits'))[0]).rotate()
        azi_map_data=azi_map_4098.data
        azi_map_data=cv2.resize(azi_map_data,(4096,4096),interpolation=cv2.INTER_AREA)
        azi_map=sunpy.map.Map(azi_map_data,azi_map_4098.meta)
        # 1 获取 header 参数
        phi0 = B_map.observer_coordinate.lon.to(u.rad).value
        b    = B_map.observer_coordinate.lat.to(u.rad).value
        # WCS rotation matrix
        pc = B_map.wcs.wcs.pc

        # P-angle（弧度）
        p = np.arctan2(pc[0,1], pc[0,0])

        # 2 像素坐标
        ny,nx = B_map.data.shape
        y,x = np.mgrid[0:ny,0:nx]

        # 3 skycoord
        coords = pixel_to_skycoord(x,y,B_map.wcs)

        # 4 heliographic
        hg = coords.transform_to(frames.HeliographicStonyhurst)

        phi = hg.lon.to(u.rad).value
        lam = hg.lat.to(u.rad).value
        #计算矩阵参数
        dphi = phi - phi0

        k11 = np.cos(lam)*(np.sin(b)*np.sin(p)*np.cos(dphi) + np.cos(p)*np.sin(dphi)) \
            - np.sin(lam)*(np.cos(b)*np.sin(p))

        k12 = -np.cos(lam)*(np.sin(b)*np.cos(p)*np.cos(dphi) - np.sin(p)*np.sin(dphi)) \
            + np.sin(lam)*(np.cos(b)*np.cos(p))

        k13 = np.cos(lam)*np.cos(b)*np.cos(dphi) + np.sin(lam)*np.sin(b)


        k21 = np.sin(lam)*(np.sin(b)*np.sin(p)*np.cos(dphi) + np.cos(p)*np.sin(dphi)) \
            + np.cos(lam)*(np.cos(b)*np.sin(p))

        k22 = -np.sin(lam)*(np.sin(b)*np.cos(p)*np.cos(dphi) - np.sin(p)*np.sin(dphi)) \
            - np.cos(lam)*(np.cos(b)*np.cos(p))

        k23 = np.sin(lam)*np.cos(b)*np.cos(dphi) - np.cos(lam)*np.sin(b)


        k31 = -np.sin(b)*np.sin(p)*np.sin(dphi) + np.cos(p)*np.cos(dphi)

        k32 =  np.sin(b)*np.cos(p)*np.sin(dphi) + np.sin(p)*np.cos(dphi)

        k33 = -np.cos(b)*np.sin(dphi)
        #这个Bx,By,Bz应该就是mapbx,mapby,mapbz，不知道为什么丢了一块
        Bx,By,Bz=mapbx,mapby,mapbz
        Br =(k11*Bx + k12*By + k13*Bz)
        Btheta = k21*Bx + k22*By + k23*Bz
        Bphi = k31*Bx + k32*By + k33*Bz
        Br_clean = Br.copy()
        Bphi_clean = Bphi.copy()
        Btheta_clean = Btheta.copy()
        Br_clean = Br_clean.astype(np.float32)


        limit = 10000   # 可以按需要改，比如 3000、5000、10000
        bad = (~np.isfinite(Br_clean)) | (np.abs(Br_clean) > limit)
        Br_clean[bad] = np.nan
        meta=B_map.meta.copy()
        meta.pop('BLANK',None)

        bad = (~np.isfinite(Bphi_clean)) | (np.abs(Bphi_clean) > limit)
        Bphi_clean[bad] = np.nan
        meta=B_map.meta.copy()
        meta.pop('BLANK',None)

        bad = (~np.isfinite(Btheta_clean)) | (np.abs(Btheta_clean) > limit)
        Btheta_clean[bad] = np.nan
        meta=B_map.meta.copy()
        meta.pop('BLANK',None)

        Br_map = sunpy.map.Map(Br_clean, meta)
        Br_map.save(os.path.join(hmi_path,'Br.fits'),overwrite=True)

            # 计算水平磁场
        Bh = np.sqrt(Btheta_clean**2 + Bphi_clean**2)

        # 计算磁场倾角 (相对于径向方向)
        inclination = np.abs(np.degrees(np.arctan2(Bh, np.abs(Br))))

        # 清理异常值
        Bh_clean = Bh.copy()
        inc_clean = inclination.copy()
        Bh_clean =Bh_clean.astype(np.float32)
        inc_clean = inc_clean.astype(np.float32)

        bad = (~np.isfinite(Bh_clean)) | (np.abs(Bh_clean) > limit)
        Bh_clean[bad] = np.nan

        bad = (~np.isfinite(inc_clean))
        inc_clean[bad] = np.nan

        # 保存 Bh
        Bh_map = sunpy.map.Map(Bh_clean, meta)
        Bh_map.save(os.path.join(hmi_path,'Bh.fits'), overwrite=True)

        # 保存 inclination
        inc_map = sunpy.map.Map(inc_clean, meta)
        inc_map.save(os.path.join(hmi_path,'inclination_calc.fits'), overwrite=True)
    except:
        print(f'{i} failed to process.')

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
D:\C\20150313_073200_UTC\hmi.B_720s failed to process.
111
112
113
114
115
116
117
118
119
120
121
122
123
124
D:\C\20150916_190800_UTC\hmi.B_720s failed to process.
125
126
127
128
129
130
131
132
133
134
135
136
137
D:\C\20151228_001800_UTC\hmi.B_720s failed to process.
138
139
140
141


In [7]:
from sunpy.coordinates import propagate_with_solar_surface

In [8]:
import pandas as pd

In [6]:
#裁剪Br incli并对目标区域进行旋转对齐+裁剪,采用131的第一张图对其进行对齐
root_dir=r'D:\C'
file_time=os.listdir(root_dir)

df=pd.read_excel('../dataset/C级耀斑.xlsx')
#df.drop(df.index[8:16], inplace=True)
x_range=df['X, arcsec'].values
y_range=df['Y, arcsec'].values
cnt=0
width=800
for i in range(len(file_time)):
    cnt=cnt+1
    print(cnt)
    #画出每个事件最早的131并进行裁剪
    try:
        ref_dir=os.path.join(root_dir,file_time[i],'131sub_map')
        ref_list=glob.glob(os.path.join(ref_dir,'*.fits'))
        map0=sunpy.map.Map(ref_list[0])
        # sub_map0=map0
        bl1=SkyCoord((x_range[i]-width//2)*u.arcsec,(y_range[i]-width//2)*u.arcsec, frame=map0.coordinate_frame)
        tr1=SkyCoord((x_range[i]+width//2)*u.arcsec,(y_range[i]+width//2)*u.arcsec, frame=map0.coordinate_frame)
        sub_map0=map0.submap(bl1,top_right=tr1)

        map1=sunpy.map.Map(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br.fits'))
        map2=sunpy.map.Map(os.path.join(root_dir,file_time[i],'hmi.B_720s','inclination_calc.fits'))
        with propagate_with_solar_surface():
            rot_sub_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
            rot_sub_map2=map2.reproject_to(sub_map0.wcs,preserve_date_obs=True)
        rot_sub_map1.meta.pop('BLANK', None)
        rot_sub_map2.meta.pop('BLANK', None)

        rot_sub_map1.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','Br_sub.fits'),overwrite=True)
        rot_sub_map2.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','inclination_calc_sub.fits'),overwrite=True)
    except Exception as e:
        print(e)


NameError: name 'pd' is not defined